In [1]:
import pandas as pd
import csv

In [10]:
def parse_params(params_str):
    params = {}
    for item in params_str.split():
        if '=' in item:
            key, value = item.split('=', 1)
            params[key] = value
    return params

# aggregates results over all parameters except those in params_to_compare
def results_csv_to_df(csv_file, params_to_compare):
    results = []

    with open(csv_file, newline='') as f:
        reader = csv.DictReader(f, fieldnames=[
            'accuracy', 'correctly_filled_cells', 'nfe', 'time', 'speed', 'not_fully_unmasked',
            'checkpoint', 'strategy', 'params'
        ])
        reader.__next__()  # Skip header row
        for row in reader:
            params = parse_params(row['params'])
            relevant_params = {k: v for k, v in params.items() if k in params_to_compare}
            row_dict = {
                'accuracy': float(row['accuracy']),
                'correctly_filled_cells': float(row['correctly_filled_cells']),
                'checkpoint': row['checkpoint'],
                'strategy': row['strategy'],
                'nfe': float(row['nfe']),
                'time': float(row['time']),
                'speed': float(row['speed']),
            }
            row_dict.update(relevant_params)
            results.append(row_dict)
    
    results_df = pd.DataFrame(results)
    aggregated_df = results_df.groupby(params_to_compare + ['checkpoint', 'strategy']).agg(
        accuracy_mean=pd.NamedAgg(column='accuracy', aggfunc='mean'),
        accuracy_std=pd.NamedAgg(column='accuracy', aggfunc='std'),
        correctly_filled_cells_mean=pd.NamedAgg(column='correctly_filled_cells', aggfunc='mean'),
        correctly_filled_cells_std=pd.NamedAgg(column='correctly_filled_cells', aggfunc='std'),
        nfe=pd.NamedAgg(column='nfe', aggfunc='mean'),
        time=pd.NamedAgg(column='time', aggfunc='mean'),
        speed=pd.NamedAgg(column='speed', aggfunc='mean'),
    ).reset_index()

    return aggregated_df

In [11]:
params_to_compare=['steps_before_pruning', 'pruning_num_beams', 'branching_factor', 'score_time', 'score_method']
df = results_csv_to_df('results_steps_before_pruning.csv', params_to_compare)
df

,steps_before_pruning,pruning_num_beams,branching_factor,score_time,score_method,checkpoint,strategy,accuracy_mean,accuracy_std,correctly_filled_cells_mean,correctly_filled_cells_std,nfe,time,speed
0,16,1,2,inferred,avg,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,0.8438,NaN,0.9353,NaN,82.95,76.0,0.84
1,16,1,2,inferred,min,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,0.8594,NaN,0.9356,NaN,82.38,73.0,0.88
2,16,2,2,inferred,avg,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,0.8594,NaN,0.9368,NaN,89.95,81.0,0.79
3,16,2,2,inferred,min,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,0.8594,NaN,0.9360,NaN,90.11,82.0,0.79
4,16,2,4,inferred,avg,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,0.8594,NaN,0.9355,NaN,178.12,157.0,0.41
5,16,2,4,inferred,min,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,0.8750,NaN,0.9389,NaN,176.06,157.0,0.41
6,16,4,4,inferred,avg,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,0.8750,NaN,0.9406,NaN,209.25,187.0,0.34
7,16,4,4,inferred,min,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,0.8750,NaN,0.9395,NaN,210.58,185.0,0.35
8,2,1,2,inferred,avg,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,0.7969,NaN,0.9156,NaN,152.72,127.0,0.50
9,2,1,2,inferred,min,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,0.7812,NaN,0.9104,NaN,155.73,133.0,0.48


In [ ]:
# compare steps_before_pruning accuracy
steps_before_pruning = df.groupby(['checkpoint', 'strategy'] + [param for param in params_to_compare if param != 'steps_before_pruning'] + ['steps_before_pruning'])['accuracy_mean'].mean().unstack()
# steps_before_pruning = df.groupby(['checkpoint', 'strategy'] + ['steps_before_pruning'])['accuracy_mean'].mean().unstack()
steps_before_pruning

steps_before_pruning                                                                                                                 16  \
checkpoint                      strategy                             pruning_num_beams branching_factor score_time score_method           
checkpoints/gidd_0_2/300_epochs gidd_change_low_confidence_positions 1                 2                inferred   avg           0.8438   
                                                                                                                   min           0.8594   
                                                                     2                 2                inferred   avg           0.8594   
                                                                                                                   min           0.8594   
                                                                                       4                inferred   avg           0.8594   
                                                                                                                   min           0.8750   
                                                                     4                 4                inferred   avg           0.8750   
                                                                                                                   min           0.8750   

steps_before_pruning                                                                                                                  2  \
checkpoint                      strategy                             pruning_num_beams branching_factor score_time score_method           
checkpoints/gidd_0_2/300_epochs gidd_change_low_confidence_positions 1                 2                inferred   avg           0.7969   
                                                                                                                   min           0.7812   
                                                                     2                 2                inferred   avg           0.8906   
                                                                                                                   min           0.8594   
                                                                                       4                inferred   avg           0.7812   
                                                                                                                   min           0.7500   
                                                                     4                 4                inferred   avg           0.7656   
                                                                                                                   min           0.8438   

steps_before_pruning                                                                                                                  4  \
checkpoint                      strategy                             pruning_num_beams branching_factor score_time score_method           
checkpoints/gidd_0_2/300_epochs gidd_change_low_confidence_positions 1                 2                inferred   avg           0.7656   
                                                                                                                   min           0.7812   
                                                                     2                 2                inferred   avg           0.8750   
                                                                                                                   min           0.8438   
                                                                                       4                inferred   avg           0.8750   
                                                                                                                   min           0.8594   
                                                                     4                 4                in

In [18]:
# compare steps_before_pruning nfe
steps_before_pruning = df.groupby(['checkpoint', 'strategy'] + [param for param in params_to_compare if param != 'steps_before_pruning'] + ['steps_before_pruning'])['nfe'].mean().unstack()
# steps_before_pruning = df.groupby(['checkpoint', 'strategy'] + ['steps_before_pruning'])['nfe'].mean().unstack()
steps_before_pruning

steps_before_pruning                                                                                                                 16  \
checkpoint                      strategy                             pruning_num_beams branching_factor score_time score_method           
checkpoints/gidd_0_2/300_epochs gidd_change_low_confidence_positions 1                 2                inferred   avg            82.95   
                                                                                                                   min            82.38   
                                                                     2                 2                inferred   avg            89.95   
                                                                                                                   min            90.11   
                                                                                       4                inferred   avg           178.12   
                                                                                                                   min           176.06   
                                                                     4                 4                inferred   avg           209.25   
                                                                                                                   min           210.58   

steps_before_pruning                                                                                                                   2  \
checkpoint                      strategy                             pruning_num_beams branching_factor score_time score_method            
checkpoints/gidd_0_2/300_epochs gidd_change_low_confidence_positions 1                 2                inferred   avg            152.72   
                                                                                                                   min            155.73   
                                                                     2                 2                inferred   avg            217.84   
                                                                                                                   min            227.09   
                                                                                       4                inferred   avg            599.11   
                                                                                                                   min            629.92   
                                                                     4                 4                inferred   avg           1088.66   
                                                                                                                   min           1096.95   

steps_before_pruning                                                                                                                  4  \
checkpoint                      strategy                             pruning_num_beams branching_factor score_time score_method           
checkpoints/gidd_0_2/300_epochs gidd_change_low_confidence_positions 1                 2                inferred   avg           119.05   
                                                                                                                   min           117.78   
                                                                     2                 2                inferred   avg           154.70   
                                                                                                                   min           158.27   
                                                                                       4                inferred   avg           387.48   
                                                                                                                   min           392.86   
                                                                     4                 4        

In [20]:
# compare steps_before_pruning time
steps_before_pruning = df.groupby(['checkpoint', 'strategy'] + [param for param in params_to_compare if param != 'steps_before_pruning'] + ['steps_before_pruning'])['time'].mean().unstack()
# steps_before_pruning = df.groupby(['checkpoint', 'strategy'] + ['steps_before_pruning'])['time'].mean().unstack()
steps_before_pruning

steps_before_pruning                                                                                                                16  \
checkpoint                      strategy                             pruning_num_beams branching_factor score_time score_method          
checkpoints/gidd_0_2/300_epochs gidd_change_low_confidence_positions 1                 2                inferred   avg            76.0   
                                                                                                                   min            73.0   
                                                                     2                 2                inferred   avg            81.0   
                                                                                                                   min            82.0   
                                                                                       4                inferred   avg           157.0   
                                                                                                                   min           157.0   
                                                                     4                 4                inferred   avg           187.0   
                                                                                                                   min           185.0   

steps_before_pruning                                                                                                                 2  \
checkpoint                      strategy                             pruning_num_beams branching_factor score_time score_method          
checkpoints/gidd_0_2/300_epochs gidd_change_low_confidence_positions 1                 2                inferred   avg           127.0   
                                                                                                                   min           133.0   
                                                                     2                 2                inferred   avg           185.0   
                                                                                                                   min           199.0   
                                                                                       4                inferred   avg           497.0   
                                                                                                                   min           512.0   
                                                                     4                 4                inferred   avg           778.0   
                                                                                                                   min           782.0   

steps_before_pruning                                                                                                                 4  \
checkpoint                      strategy                             pruning_num_beams branching_factor score_time score_method          
checkpoints/gidd_0_2/300_epochs gidd_change_low_confidence_positions 1                 2                inferred   avg           105.0   
                                                                                                                   min           104.0   
                                                                     2                 2                inferred   avg           153.0   
                                                                                                                   min           140.0   
                                                                                       4                inferred   avg           335.0   
                                                                                                                   min           339.0   
                                                                     4                 4                inferred   avg           498.0

In [21]:
# Get best score_method and steps_before_pruning per pruning_num_beams and branching_factor
best_method_and_steps_before_pruning = df.loc[df.groupby(['checkpoint', 'strategy', 'pruning_num_beams', 'branching_factor', 'score_time'])['accuracy_mean'].idxmax()]#.reset_index(drop=True)
best_method_and_steps_before_pruning[['checkpoint', 'strategy', 'pruning_num_beams', 'branching_factor', 'score_time', 'score_method', 'steps_before_pruning', 'accuracy_mean', 'nfe', 'speed']]

,checkpoint,strategy,pruning_num_beams,branching_factor,score_time,score_method,steps_before_pruning,accuracy_mean,nfe,speed
1,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,1,2,inferred,min,16,0.8594,82.38,0.88
10,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,2,2,inferred,avg,2,0.8906,217.84,0.35
28,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,2,4,inferred,avg,8,0.9062,246.14,0.29
22,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,4,4,inferred,avg,4,0.9375,608.12,0.13
